In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os
from pathlib import Path
from datetime import datetime

project_root = Path.cwd().parent.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

from src.features.features_v1 import *
from src.features.features_v2 import *
from src.pipeline.calculate_evs import *
from src.utils.helper_functions import *
from src.utils.team_info import teamStarPlayer, projectedStartingFive, mainStartingFive

### Update projected starting lineups

In [2]:
from src.utils.scrap_starters import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
scraper.updateTeamInfo()  # Update teamInfo.py

Successfully updated /Users/alexgonzalez/Documents/NBA-Prop-Predictor/src/utils/team_info.py
Updated 14 teams with confirmed lineups


### Load Model

### Load Player Data and Bookmaker Data

In [3]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

def get_latest_file(pattern):
    files = list(Path('data/raw/player_lines').glob(pattern))
    if not files:
        return None
    return max(files, key=lambda p: p.stat().st_mtime)

us_file = get_latest_file(f'NBA_US_{today}*.csv')
dfs_file = get_latest_file(f'NBA_DFS_{today}*.csv')

if us_file is None:
    raise FileNotFoundError(f"No NBA_US file found for {today}")
if dfs_file is None:
    raise FileNotFoundError(f"No NBA_DFS file found for {today}")

s26 = pd.read_csv('data/processed/training/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')
usData = pd.read_csv(us_file)
dfsData = pd.read_csv(dfs_file)

print(f"Loaded: {us_file.name}")
print(f"Loaded: {dfs_file.name}")
dfsData.head()

Loaded: NBA_US_20251207_104654.csv
Loaded: NBA_DFS_20251207_104754.csv


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE,DATA_PULLED_AT
0,Betr DFS,player_points,Josh Hart,Over,18.5,-137,2025-12-07,2025-12-07T18:47:15Z,2025-12-07 10:47:54
1,Betr DFS,player_points,Josh Hart,Under,18.5,-137,2025-12-07,2025-12-07T18:47:15Z,2025-12-07 10:47:54
2,Betr DFS,player_points,Mikal Bridges,Over,14.5,-137,2025-12-07,2025-12-07T18:47:15Z,2025-12-07 10:47:54
3,Betr DFS,player_points,Mikal Bridges,Under,14.5,-137,2025-12-07,2025-12-07T18:47:15Z,2025-12-07 10:47:54
4,Betr DFS,player_points,Jaylen Brown,Over,29.5,-137,2025-12-07,2025-12-07T18:47:15Z,2025-12-07 10:47:54


In [4]:
from src.features.feature_engine import FeatureEngine

engine = FeatureEngine({
    "min_model": "src/models/saved/min_model.pkl",
    "usg_model": "src/models/saved/usg_model.pkl",
    "fga_model": "src/models/saved/fga_model.pkl",
    "ngboost_model_wrapper": "src/models/saved/pts_model_wrapper.pkl"
})

/Users/alexgonzalez/Documents/NBA-Prop-Predictor/nba_model/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Top EVs for 2 leg bets

### Underdog picks

In [5]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

underdogPairs = calculate2LegBets(
    data=s26,
    bookmakers=dfsPTS,
    engine=engine,
    current_date=current_date,
    top_n=10,                    # Get top 10 bets
    max_player_appearances=1,    # Each player appears max once
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)

underdogPairs = underdogPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2','ODDS 1', 'ODDS 2','PREDICTION 1', 'PREDICTION 2', 'MODEL_PROB 1', 'MODEL_PROB 2', 'SIDE 1', 'SIDE 2', 'PARLAY_PROB', 'PARLAY_ODDS', 'EV_PERCENT', 'KELLY_QUARTER']]
underdogPairs.to_csv('data/props/ev_analysis/underdogPairs.csv', index=False)
underdogPairs

Computing predictions for 53 players...
[MIN] No data found for Vincent Williams Jr.
Found 52 valid players
Generated 1108 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,ODDS 1,ODDS 2,PREDICTION 1,PREDICTION 2,MODEL_PROB 1,MODEL_PROB 2,SIDE 1,SIDE 2,PARLAY_PROB,PARLAY_ODDS,EV_PERCENT,KELLY_QUARTER
678,KJ Simpson,VJ Edgecombe,11.5,10.5,-125,110,3.09,14.51,0.912,0.806,under,over,0.735,278,178.01,0.1601
654,Spencer Jones,Jaylin Williams,6.5,8.5,-137,-127,1.21,3.08,0.923,0.822,under,under,0.758,209,134.27,0.1606
551,Cameron Johnson,Ace Bailey,13.5,12.5,114,-115,8.83,6.91,0.719,0.772,under,under,0.555,300,121.94,0.1016
516,Brandon Miller,Jalen Williams,20.5,23.5,115,-110,15.33,16.70,0.710,0.746,under,under,0.529,310,116.88,0.0943
398,Jordan Walsh,Josh Giddey,7.5,21.5,-108,-104,3.88,16.98,0.717,0.696,under,under,0.499,278,88.54,0.0796
600,Peyton Watson,Jimmy Butler III,12.5,20.5,-104,105,8.53,16.52,0.681,0.644,under,under,0.438,302,76.22,0.0631
743,Deni Avdija,Will Richard,25.5,8.5,-119,-114,20.30,4.61,0.711,0.696,under,under,0.495,245,70.91,0.0724
425,Nikola Jokić,Santi Aldama,29.5,11.5,-118,-105,23.79,13.00,0.693,0.646,under,over,0.448,261,61.55,0.0590
924,Cam Spencer,Keyonte George,11.5,21.5,100,-107,8.49,17.69,0.605,0.647,under,under,0.391,287,51.47,0.0448
1057,Tyrese Maxey,Aaron Wiggins,27.5,15.5,100,-104,28.28,17.14,0.580,0.611,over,over,0.354,292,38.81,0.0332


### Prizepicks picks

In [6]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

prizepicksPairs = calculate2LegBets(
    data=s26,
    bookmakers=dfsPTS,
    engine=engine,
    current_date=current_date,
    top_n=10,                    # Get top 10 bets
    max_player_appearances=1,    # Each player appears max once
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)

prizepicksPairs = prizepicksPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2','ODDS 1', 'ODDS 2','PREDICTION 1', 'PREDICTION 2', 'MODEL_PROB 1', 'MODEL_PROB 2', 'SIDE 1', 'SIDE 2', 'PARLAY_PROB', 'PARLAY_ODDS', 'EV_PERCENT', 'KELLY_QUARTER']]
prizepicksPairs.to_csv('data/props/ev_analysis/prizepicksPairs.csv', index=False)
prizepicksPairs

Computing predictions for 71 players...
[MIN] No data found for Vincent Williams Jr.
Found 70 valid players
Generated 2026 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,ODDS 1,ODDS 2,PREDICTION 1,PREDICTION 2,MODEL_PROB 1,MODEL_PROB 2,SIDE 1,SIDE 2,PARLAY_PROB,PARLAY_ODDS,EV_PERCENT,KELLY_QUARTER
1244,Tidjane Salaün,Jake LaRavia,7.5,4.5,101,-120,2.17,9.42,0.841,0.906,under,over,0.762,268,180.41,0.1683
635,Ja'Kobe Walter,KJ Simpson,7.5,11.5,110,-125,2.91,3.09,0.765,0.912,under,under,0.697,278,163.57,0.1471
998,Cameron Johnson,Kenrich Williams,13.5,7.5,114,-122,8.83,2.55,0.719,0.836,under,under,0.602,289,134.00,0.1159
855,Brandon Miller,VJ Edgecombe,21.5,10.0,-108,-137,15.33,14.51,0.753,0.837,under,over,0.630,233,109.92,0.1179
622,Jordan Walsh,Ace Bailey,7.5,12.5,-108,-115,3.88,6.91,0.717,0.772,under,under,0.553,260,99.03,0.0952
388,Anfernee Simons,Jalen Williams,11.5,23.5,-110,-110,14.38,16.70,0.717,0.746,over,under,0.535,264,94.68,0.0897
1776,Josh Giddey,Jaylin Williams,21.5,8.0,-104,-137,16.98,3.08,0.696,0.798,under,under,0.555,239,88.25,0.0923
1038,Peyton Watson,Jusuf Nurkić,12.5,9.5,-104,112,8.53,6.07,0.681,0.621,under,under,0.423,316,75.98,0.0601
1349,Deni Avdija,Will Richard,25.5,8.5,-119,-114,20.30,4.61,0.711,0.696,under,under,0.495,245,70.91,0.0724
743,Nikola Jokić,Jaren Jackson Jr.,29.5,19.5,-118,105,23.79,15.99,0.693,0.634,under,under,0.439,279,66.30,0.0594


## 3 leg parlay

### Underdog picks

In [7]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points') ]

underdogTrios = calculate3LegBets(
    data=s26,
    bookmakers=dfsPTS,
    engine=engine,
    current_date=current_date,
    top_n=10,                    # Get top 10 bets
    max_player_appearances=1,    # Each player appears max once
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)

underdogTrios = underdogTrios[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'ODDS 1', 'ODDS 2', 'ODDS 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'MODEL_PROB 1', 'MODEL_PROB 2', 'MODEL_PROB 3', 'SIDE 1', 'SIDE 2', 'SIDE 3', 'PARLAY_PROB', 'PARLAY_ODDS', 'EV_PERCENT', 'KELLY_QUARTER']]
underdogTrios.to_csv('data/props/ev_analysis/underdogTrios.csv', index=False)
underdogTrios.head()

Computing predictions for 53 players...
[MIN] No data found for Vincent Williams Jr.
Found 52 valid players
Generated 12352 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,ODDS 1,ODDS 2,ODDS 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,MODEL_PROB 1,MODEL_PROB 2,MODEL_PROB 3,SIDE 1,SIDE 2,SIDE 3,PARLAY_PROB,PARLAY_ODDS,EV_PERCENT,KELLY_QUARTER
10164,KJ Simpson,VJ Edgecombe,Jaylin Williams,11.5,10.5,8.5,-125,110,-127,3.09,14.51,3.08,0.912,0.806,0.822,under,over,under,0.604,576,308.58,0.1339
6425,Jordan Walsh,Spencer Jones,Ace Bailey,7.5,6.5,12.5,-108,-137,-115,3.88,1.21,6.91,0.717,0.923,0.772,under,under,under,0.510,523,217.76,0.1041
8422,Cameron Johnson,Josh Giddey,Jalen Williams,13.5,21.5,23.5,114,-104,-110,8.83,16.98,16.70,0.719,0.696,0.746,under,under,under,0.373,701,198.93,0.0709
7760,Brandon Miller,Deni Avdija,Jimmy Butler III,20.5,25.5,20.5,115,-119,105,15.33,20.30,16.52,0.710,0.711,0.644,under,under,under,0.325,711,163.60,0.0575
9083,Peyton Watson,Santi Aldama,Will Richard,12.5,11.5,8.5,-104,-105,-114,8.53,13.00,4.61,0.681,0.646,0.696,under,over,under,0.306,619,120.25,0.0486


### Prizepicks picks

In [8]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

triosPrizepicks = calculate3LegBets(
    data=s26,
    bookmakers=dfsPTS,
    engine=engine,
    current_date=current_date,
    top_n=10,                    # Get top 10 bets
    max_player_appearances=1,    # Each player appears max once
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)

triosPrizepicks = triosPrizepicks[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'ODDS 1', 'ODDS 2', 'ODDS 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'MODEL_PROB 1', 'MODEL_PROB 2', 'MODEL_PROB 3', 'SIDE 1', 'SIDE 2', 'SIDE 3', 'PARLAY_PROB', 'PARLAY_ODDS', 'EV_PERCENT', 'KELLY_QUARTER']]
triosPrizepicks.to_csv('data/props/ev_analysis/prizepicksTrios.csv', index=False)
triosPrizepicks.head()

Computing predictions for 71 players...
[MIN] No data found for Vincent Williams Jr.
Found 70 valid players
Generated 31000 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,ODDS 1,ODDS 2,ODDS 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,MODEL_PROB 1,MODEL_PROB 2,MODEL_PROB 3,SIDE 1,SIDE 2,SIDE 3,PARLAY_PROB,PARLAY_ODDS,EV_PERCENT,KELLY_QUARTER
14638,Ja'Kobe Walter,Tidjane Salaün,Jake LaRavia,7.5,7.5,4.5,110,101,-120,2.91,2.17,9.42,0.765,0.841,0.906,under,under,over,0.583,674,350.89,0.1302
22818,KJ Simpson,VJ Edgecombe,Kenrich Williams,11.5,10.0,7.5,-125,-137,-122,3.09,14.51,2.55,0.912,0.837,0.836,under,over,under,0.638,467,262.01,0.1403
13103,Jordan Walsh,Cameron Johnson,Ace Bailey,7.5,13.5,12.5,-108,114,-115,3.88,8.83,6.91,0.717,0.719,0.772,under,under,under,0.398,671,206.54,0.0770
7828,Anfernee Simons,Brandon Miller,Jalen Williams,11.5,21.5,23.5,-110,-108,-110,14.38,15.33,16.70,0.717,0.753,0.746,over,under,under,0.403,602,182.81,0.0759
21236,Peyton Watson,Josh Giddey,Jaylin Williams,12.5,21.5,8.0,-104,-104,-137,8.53,16.98,3.08,0.681,0.696,0.798,under,under,under,0.378,566,151.75,0.0670
